# Figure 2 - RNA parts mostly <br>
## Some supplementary panels for supplementary figure 2 and 3 are also included

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import matplotlib
import seaborn as sns
import textwrap
import numpy as np
from scipy import stats
import glob 
import os
import datetime
import anndata as ad
from scipy import sparse
from anndata import AnnData


matplotlib.rcParams['pdf.fonttype'] = 42

In [ ]:
# dictionaries

# cell type name correction
cell_type_labels_corr={"L2_3_IT": "L2/3 IT","L5_IT": "L5 IT","L5_ET":"L5 ET", "L6_CT": "L6 CT", "L4_IT": "L4 IT", "L5_IT_A": "L5 IT A", "L5_IT_B": "L5 IT B", "L6_IT": "L6 IT", "L6_IT_Car3": "L6 IT Car3", "Sst_Chodl":"Sst Chodl","L5_6_NP": "L5/6 NP","Lamp5_Exc":"Lamp5 Exc", "neurons" : "Neurons", "immune":"Immune", "neurons_unk" : "Neurons unk"}
dict_colors =  {'L2/3 IT': '#ff0000',
 'Oligo': '#149900',
 'Astro': '#8f00b3',
 'OPC': '#2EF0F0',
 'Micro-PVM': '#bf9360',
 'L5 IT': '#B2F7B7',
 'Vip': '#79baf2',
 'Pvalb': '#FA9A6E',
 'Sst': '#990000',
 'L6 IT': '#F7A400',
 'Lamp5': '#2d4459',
 'L4 IT': '#307cbf',
 'L6 CT': '#4c0000',
 'Sncg': '#FA7ADC',
 'L6b': '#00f220',
 'L5/6 NP': '#accbe6',
 'L5 ET': '#520066',
 'L6 IT Car3': '#330000',
 'VLMC': '#594f43',
 'Endo': '#16591f',
 'Sst Chodl': '#00FF00',
 'Neurons': '#999545',
 'DopaN': '#cc3333',
 'Lamp5 Exc': '#e59900',
 'GabaN': '#ace6b4',
 'OLG/OPC': '#262d33',
 'neuron_unk': '#ee00ff',
 'GlutaN': '#e57373',
 'unk': '#8c5e00',
 'Neurons unk':'#8c5e00',
'Immune':'deepskyblue',
 'L5 IT B':'#B2F7B7',
'L5 IT A':'palegreen',
}

# UMAP

In [ ]:
path = '../../snRNA/01_analysis/02_combination_10x_pb/hdf5/cc_rna.h5ad' # can be replaced with csv as well
adata_cc = sc.read_h5ad(path) 
path = '../../snRNA/01_analysis/02_combination_10x_pb/hdf5/sn_rna.h5ad'
adata_sn = sc.read_h5ad(path)

In [ ]:
cc_rna = pd.DataFrame(adata_cc.obsm['X_umap'])
df1 = adata_cc.obs['consensus_cell_type_corrected']
df1 = df1.reset_index()
cc_rna = pd.concat([cc_rna, df1], axis = 1, ignore_index = True)
cc_rna = cc_rna.rename(columns={0: "UMAP_1", 1: "UMAP_2", 2:"index", 3:'cell_type'})
cc_rna['correct_ct'] = [cell_type_labels_corr.get(x, x) for x in cc_rna['cell_type']]
cc_rna['color'] = cc_rna['correct_ct'].map(dict_colors)
cc_rna.head()

In [ ]:
sn_rna = pd.DataFrame(adata_sn.obsm['X_umap'])
df1 = adata_sn.obs['cell_type_neuron_incl']
df1 = df1.reset_index()
sn_rna = pd.concat([sn_rna, df1], axis = 1, ignore_index = True)
sn_rna = sn_rna.rename(columns={0: "UMAP_1", 1: "UMAP_2", 2:"index", 3:'cell_type'})
sn_rna['correct_ct'] = [cell_type_labels_corr.get(x, x) for x in sn_rna['cell_type']]
sn_rna['color'] = sn_rna['correct_ct'].map(dict_colors)
sn_rna.head()

In [ ]:
cc_atac = 'ATAC_CC_metadata_coordinates.csv' # change path if needed
cc_atac = pd.read_csv(cc_atac)
sn_atac = 'ATAC_SN_metadata_coordinates.csv'
sn_atac = pd.read_csv(sn_atac)

In [ ]:
cc_atac['correct_ct'] = [cell_type_labels_corr.get(x, x) for x in cc_atac['CISTOPIC_2_final_annot']]
sn_atac['correct_ct'] = [cell_type_labels_corr.get(x, x) for x in sn_atac['CISTOPIC_2_final_annot']]
cc_atac['color'] = cc_atac['correct_ct'].map(dict_colors)
sn_atac['color'] = sn_atac['correct_ct'].map(dict_colors)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(30, 20))
dataframes = [sn_rna, sn_atac, cc_rna, cc_atac]  


for ax, df in zip(axes.flat, dataframes):
    present_labels = sorted([label for label in df['correct_ct'].unique() if label in dict_colors])
    ax.scatter(x=df['UMAP_1'], y=df['UMAP_2'], c=df['color'], s=2, rasterized=True)
    handles = [
        plt.Line2D([0], [0], marker='o', color=dict_colors[label], linestyle='', markersize=10, label=label)
        for label in present_labels
    ]
    ax.legend(handles=handles, title='Cell type', bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False, fontsize = 15, title_fontsize=15 )
    ax.axis('off')
    ax.set_frame_on(False)
   
plt.tight_layout()
plt.savefig('figures/umap_all.pdf')
plt.show()

# cell count barplots - supplementary 2

In [ ]:
# substantia nigra
rna_counts = pd.concat(
    [
        sn_rna["correct_ct"].value_counts().rename("sn_rna"),
    ],
    axis=1
).fillna(0).astype(int)

atac_counts = pd.concat(
    [
        sn_atac["correct_ct"].value_counts().rename("sn_atac"),
    ],
    axis=1
).fillna(0).astype(int)

all_categories = sorted(set(rna_counts.index).union(atac_counts.index))

rna_counts = rna_counts.reindex(all_categories, fill_value=0)
atac_counts = atac_counts.reindex(all_categories, fill_value=0)

all_categories = rna_counts["sn_rna"].sort_values(ascending=True).index.tolist()

rna_counts = rna_counts.loc[all_categories]
atac_counts = atac_counts.loc[all_categories]

color_map = (
    pd.concat([
        sn_rna[["correct_ct", "color"]],
        sn_atac[["correct_ct", "color"]],
    ])
    .dropna()
    .drop_duplicates(subset=["correct_ct"])
    .set_index("correct_ct")["color"]
    .to_dict()
)

bar_colors = [color_map.get(category, "#808080") for category in all_categories]

fig, axes = plt.subplots(1, 2, figsize=(8, 8), sharey=True)

axes[0].barh(
    all_categories,
    rna_counts["sn_rna"],
    color=bar_colors,
    edgecolor="black",
    linewidth=0.4
)

axes[1].barh(
    all_categories,
    atac_counts["sn_atac"],
    color=bar_colors,
    edgecolor="black",
    linewidth=0.4
)

axes[0].set_title("snRNA")
axes[1].set_title("snATAC")

axes[0].set_xlabel("Cell count")
axes[1].set_xlabel("Cell count")
axes[0].set_ylabel("Cell type")
axes[1].set_ylabel("")

# Use symlog so categories with zero cells in one modality are still handled cleanly.
axes[0].set_xscale("symlog", linthresh=1)
axes[1].set_xscale("symlog", linthresh=1)

axes[0].grid(False)
axes[1].grid(False)

plt.tight_layout()
plt.savefig("figures/manuscript_cell_counts_sn.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# cingulate cortex
rna_counts = pd.concat(
    [
        cc_rna["correct_ct"].value_counts().rename("cc_rna"),
    ],
    axis=1
).fillna(0).astype(int)

atac_counts = pd.concat(
    [
        cc_atac["correct_ct"].value_counts().rename("cc_atac"),
    ],
    axis=1
).fillna(0).astype(int)

all_categories = sorted(set(rna_counts.index).union(atac_counts.index))

rna_counts = rna_counts.reindex(all_categories, fill_value=0)
atac_counts = atac_counts.reindex(all_categories, fill_value=0)

all_categories = rna_counts["cc_rna"].sort_values(ascending=True).index.tolist()

rna_counts = rna_counts.loc[all_categories]
atac_counts = atac_counts.loc[all_categories]

color_map = (
    pd.concat([
        cc_rna[["correct_ct", "color"]],
        cc_atac[["correct_ct", "color"]],
    ])
    .dropna()
    .drop_duplicates(subset=["correct_ct"])
    .set_index("correct_ct")["color"]
    .to_dict()
)

bar_colors = [color_map.get(category, "#808080") for category in all_categories]

fig, axes = plt.subplots(1, 2, figsize=(8, 8), sharey=True)

axes[0].barh(
    all_categories,
    rna_counts["cc_rna"],
    color=bar_colors,
    edgecolor="black",
    linewidth=0.4
)

axes[1].barh(
    all_categories,
    atac_counts["cc_atac"],
    color=bar_colors,
    edgecolor="black",
    linewidth=0.4
)

axes[0].set_title("snRNA")
axes[1].set_title("snATAC")

axes[0].set_xlabel("Cell count")
axes[1].set_xlabel("Cell count")
axes[0].set_ylabel("Cell type")
axes[1].set_ylabel("")


axes[0].set_xscale("symlog", linthresh=1)
axes[1].set_xscale("symlog", linthresh=1)

axes[0].grid(False)
axes[1].grid(False)

plt.tight_layout()
plt.savefig("figures/manuscript_cell_counts_cc.pdf", dpi=300, bbox_inches="tight")
plt.show()

# Supplementary 2 methods

In [ ]:
cc_atac.loc[cc_atac['technology'] == 'HYAv2/scATAC', 'technology'] = 'scATAC'
cc_atac.loc[cc_atac['technology'] == 'scATAC/HYAv2', 'technology'] = 'scATAC'
sn_atac.loc[sn_atac['technology'] == 'HYAv2/scATAC', 'technology'] = 'scATAC'
sn_atac.loc[sn_atac['technology'] == 'scATAC/HYAv2', 'technology'] = 'scATAC'

mapper = {
    "MultiomeATAC": "10x snMultiomeATAC",
    "scATAC": "10x snATAC",
    "ScaleBioIH": "Scaled snATAC",
    "ScaleBio": "Scaled snATAC",
    "ScaleBioIH-HYAv2": "Scaled snATAC (HyDrop snATAC v2 barcodes)",
    "HYAv2": "HyDrop snATAC v2",
    "ScaleBioIH6": "Scaled snATAC",
}

cc_atac["technology"] = cc_atac["technology"].replace(mapper)
sn_atac["technology"] = sn_atac["technology"].replace(mapper)

In [ ]:
cc_atac_counts = pd.concat(
    [
        cc_atac["technology"].value_counts().rename("cc_atac"),
    ],
    axis=1
).fillna(0).astype(int)

sn_atac_counts = pd.concat(
    [
        sn_atac["technology"].value_counts().rename("sn_atac"),
    ],
    axis=1
).fillna(0).astype(int)

all_methods = sorted(set(cc_atac_counts.index).union(sn_atac_counts.index))

cc_atac_counts = cc_atac_counts.reindex(all_methods, fill_value=0)
sn_atac_counts = sn_atac_counts.reindex(all_methods, fill_value=0)

all_methods = cc_atac_counts["cc_atac"].sort_values(ascending=True).index.tolist()

cc_atac_counts = cc_atac_counts.loc[all_methods]
sn_atac_counts = sn_atac_counts.loc[all_methods]

cc_color = "#7FC97F"   
sn_color = "#BEAED4"  

# Wrap long method names to multiple lines
wrap_width = 22
wrapped_methods = [textwrap.fill(m, width=wrap_width) for m in all_methods]


fig, axes = plt.subplots(1, 2, figsize=(8, 8), sharey=True)

axes[0].barh(
    wrapped_methods,
    cc_atac_counts["cc_atac"],
    color=cc_color,
    edgecolor="black",
    linewidth=0.4
)

axes[1].barh(
    wrapped_methods,
    sn_atac_counts["sn_atac"],
    color=sn_color,
    edgecolor="black",
    linewidth=0.4
)

axes[0].set_title("CC")
axes[1].set_title("SN")

axes[0].set_xlabel("Count")
axes[1].set_xlabel("Count")
axes[0].set_ylabel("Method")
axes[1].set_ylabel("")

axes[0].grid(False)
axes[1].grid(False)

# Give more room for wrapped labels on the left
plt.tight_layout()
plt.subplots_adjust(left=0.38)


plt.tight_layout()
plt.savefig("figures/method_counts_cc_atac_sn_atac.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# methods RNA counts
cc_rna_counts = pd.concat(
    [
        adata_cc.obs["batch"].value_counts().rename("cc_rna"),
    ],
    axis=1
).fillna(0).astype(int)

sn_rna_counts = pd.concat(
    [
        adata_sn.obs["batch"].value_counts().rename("sn_rna"),
    ],
    axis=1
).fillna(0).astype(int)

all_batches = sorted(set(cc_rna_counts.index).union(sn_rna_counts.index))

cc_rna_counts = cc_rna_counts.reindex(all_batches, fill_value=0)
sn_rna_counts = sn_rna_counts.reindex(all_batches, fill_value=0)

all_batches = cc_rna_counts["cc_rna"].sort_values(ascending=True).index.tolist()

cc_rna_counts = cc_rna_counts.loc[all_batches]
sn_rna_counts = sn_rna_counts.loc[all_batches]


cc_color = "#7FC97F"   
sn_color = "#BEAED4"  

fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharey=True)

axes[0].barh(
    all_batches,
    cc_rna_counts["cc_rna"],
    color=cc_color,
    edgecolor="black",
    linewidth=0.4
)

axes[1].barh(
    all_batches,
    sn_rna_counts["sn_rna"],
    color=sn_color,
    edgecolor="black",
    linewidth=0.4
)

axes[0].set_title("CC")
axes[1].set_title("SN")

axes[0].set_xlabel("Count")
axes[1].set_xlabel("Count")
axes[0].set_ylabel("Batch")
axes[1].set_ylabel("")



axes[0].grid(False)
axes[1].grid(False)

plt.tight_layout()
plt.savefig("figures/batch_counts_cc_rna_sn_rna.pdf", dpi=300, bbox_inches="tight")
plt.show()

# Index shared nuclei between MO ATAC and RNA

In [ ]:
path = '../../snRNA/01_analysis/02_combination_10x_pb/hdf5/cc_rna.h5ad' # can be replaced with csv as well
adata_cc = sc.read_h5ad(path) 
path = '../../snRNA/01_analysis/02_combination_10x_pb/hdf5/sn_rna.h5ad'
adata_sn = sc.read_h5ad(path)

cc_atac = 'ATAC_CC_metadata_coordinates.csv' # change path if needed
cc_atac = pd.read_csv(cc_atac)
sn_atac = 'ATAC_SN_metadata_coordinates.csv'
sn_atac = pd.read_csv(sn_atac)

sn_atac['Unnamed: 0'] = sn_atac['Unnamed: 0'].str.split("___").str[0].str.upper()
sn_atac = sn_atac.set_index('Unnamed: 0')
sn_atac = sn_atac[
    sn_atac.index.str.contains(r'MO', regex=True, na=False)
]

In [ ]:
adata_sn.obs.index = adata_sn.obs.index.str.upper()
sn_rna = adata_sn.obs
sn_rna = sn_rna[
    sn_rna.index.str.contains(r'MO', regex=True, na=False)
]

In [ ]:
x_name = 'RNA'
y_name = 'ATAC'
idx_x = pd.Index(sn_rna.index).unique()
idx_y = pd.Index(sn_atac.index).unique()

shared = idx_x.intersection(idx_y)
only_x = idx_x.difference(idx_y)
only_y = idx_y.difference(idx_x)

total_unique = len(idx_x.union(idx_y))

summary_sn = {
        "x_name": x_name,
        "y_name": y_name,
        "n_x": len(idx_x),
        "n_y": len(idx_y),
        "shared": len(shared),
        f"only_{x_name}": len(only_x),
        f"only_{y_name}": len(only_y),
        "shared_ratio": len(shared) / total_unique if total_unique else 0,
        f"only_{x_name}_ratio": len(only_x) / total_unique if total_unique else 0,
        f"only_{y_name}_ratio": len(only_y) / total_unique if total_unique else 0,
}
summary_sn

In [ ]:
cc_atac['Unnamed: 0'] = cc_atac['Unnamed: 0'].str.split("___").str[0].str.upper()
cc_atac = cc_atac.set_index('Unnamed: 0')
cc_atac = cc_atac[
    cc_atac.index.str.contains(r'MO', regex=True, na=False)
]

In [ ]:
adata_cc.obs.index = adata_cc.obs.index.str.upper()
cc_rna = adata_cc.obs
cc_rna = cc_rna[
    cc_rna.index.str.contains(r'MO', regex=True, na=False)
]

In [ ]:
x_name = 'RNA'
y_name = 'ATAC'
idx_x = pd.Index(cc_rna.index).unique()
idx_y = pd.Index(cc_atac.index).unique()

shared = idx_x.intersection(idx_y)
only_x = idx_x.difference(idx_y)
only_y = idx_y.difference(idx_x)

total_unique = len(idx_x.union(idx_y))

summary_cc = {
        "x_name": x_name,
        "y_name": y_name,
        "n_x": len(idx_x),
        "n_y": len(idx_y),
        "shared": len(shared),
        f"only_{x_name}": len(only_x),
        f"only_{y_name}": len(only_y),
        "shared_ratio": len(shared) / total_unique if total_unique else 0,
        f"only_{x_name}_ratio": len(only_x) / total_unique if total_unique else 0,
        f"only_{y_name}_ratio": len(only_y) / total_unique if total_unique else 0,
}
summary_cc

In [ ]:
plot_df = pd.DataFrame(
    {
        "shared": [summary_cc["shared_ratio"], summary_sn["shared_ratio"]],
        f'only_{summary_cc["x_name"]}': [summary_cc["only_RNA_ratio"], summary_sn["only_RNA_ratio"]],
        f'only_{summary_cc["y_name"]}': [summary_cc["only_ATAC_ratio"], summary_sn["only_ATAC_ratio"]],
    },
    index=["CC", "SN"]
)

ax = plot_df.plot(
    kind="bar",
    stacked=True,
    figsize=(4, 4),
    color=["#3B8EA5", "#F4A259", "#BC4B51"],
    edgecolor="black"
)

ax.set_xlabel("")
ax.set_ylabel("Ratio")
ax.set_title("")
ax.set_ylim(0, 1)
ax.legend(title="", loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False)
ax.grid(False)

plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("figures/manuscript_stacked_barplot_shared_mo_nuclei.pdf", dpi=300, bbox_inches="tight")
plt.show()

# donor cell numbers per region and modality

In [ ]:
atac_cc = pd.DataFrame({
    "donor_id": cc_atac["donor_id"],
    "modality": "ATAC",
    "region":"CC"
})

atac_sn = pd.DataFrame({
    "donor_id": sn_atac["donor_id"],
    "modality": "ATAC",
    "region":"SN"
})

rna_cc = pd.DataFrame({
    "donor_id": adata_cc.obs["vireo_assignment"],
    "modality": "RNA",
    "region":"CC"
})


rna_sn = pd.DataFrame({
    "donor_id": adata_sn.obs["vireo_assignment"],
    "modality": "RNA",
    "region":"SN"
})



In [ ]:
combined = pd.concat([atac_cc, atac_sn, rna_cc,rna_sn], ignore_index=True)

In [ ]:
combined = combined[~combined["donor_id"].isin(["doublet", "unassigned"])]

In [ ]:
# Count occurrences per donor_id, region, modality
donor_counts = combined.groupby(['region', 'modality', 'donor_id']).size().reset_index(name='count')
donor_counts['group'] = donor_counts['region'] + ', ' + donor_counts['modality']

# region colors
region_palette = {"CC": "#7FC97F", "SN": "#BEAED4"}
group_to_region = donor_counts.set_index('group')['region'].to_dict()
unique_groups = donor_counts['group'].unique()
palette = [region_palette[group_to_region[g]] for g in unique_groups]

plt.figure(figsize=(8, 6))
sns.boxplot(x='group', y='count', data=donor_counts, showfliers=False, palette=palette)
sns.stripplot(x='group', y='count', data=donor_counts, color='black', jitter=0.3, alpha = .6)


plt.ylabel('Number of nuclei/donor', fontsize=15)
plt.xlabel('', fontsize=15)
plt.xticks(fontsize=15)
plt.yticks(fontsize=15)
plt.tight_layout()
plt.savefig('figures/numbers_nuclei_donors.pdf')

plt.show()

In [ ]:
donor_counts.groupby('group')['count'].mean()


In [ ]:
donor_counts.groupby('group')['count'].min()


In [ ]:
donor_counts.groupby('group')['count'].max()


In [ ]:
donor_counts.groupby('modality')['count'].mean()

In [ ]:
donor_counts.groupby('modality')['count'].min()

In [ ]:
donor_counts.groupby('modality')['count'].max()

# SN gene expression dotplot + QC plots

In [ ]:
adata_sn.obs['correct_ct'] = [cell_type_labels_corr.get(x, x) for x in adata_sn.obs['cell_type_neuron_incl']]
adata_cc.obs
adata_sn.X  = adata_sn.layers['log1p_norm']
marker_genes = {
    "Neurons": ["SYT1"],
    "DopaN":["TH","SLC6A3","SLC18A2"],
    "GabaN":["GAD1","GAD2"],
    "GlutaN":["SLC17A6","SLC17A7"],
     "Astro":["AQP4", "GFAP"],
    "Oligo": ["MAG", "MOBP"],
    "OPC": [
        "PDGFRA","C1QL1"],
    
    "Micro-PVM":["CD74","RUNX1"],
    "Immune":["IL7R","CD8A"],
    "Endo":["CLDN5","RBPMS","EPAS1"], 
}

dp = sc.pl.dotplot(
    adata_sn,
    marker_genes,
    groupby="correct_ct",
    standard_scale="var",
    mean_only_expressed=False,
    swap_axes=False,
    categories_order=[
        'Neurons unk',
        'DopaN',
        'GabaN',
        'GlutaN',
        'Astro',
        'Oligo',
        'OPC',
        'Micro-PVM',
        'Immune',
        'Endo'
    ],
    # save='genes_of_interest.pdf',
    # return_fig=True,
    show=False
)
ax = dp["mainplot_ax"]
# Loop through ticklabels and make them italic
for l in ax.get_xticklabels():
    l.set_style("italic")

plt.savefig('figures/sn_dotplot_marker_genes.pdf', bbox_inches="tight")
plt.show()

In [ ]:
# combine gene information from 10x and parsebio
adata_sn.obs['genes_detected'] = np.where(adata_sn.obs['gene_count'].notna(), adata_sn.obs['gene_count'], adata_sn.obs['n_genes'])
adata_sn.obs['genes_detected']

In [ ]:
def plotviolin(x,y):
    plt.figure(figsize=(5, 5))
    sns.violinplot(data=adata_sn.obs, x=x, y=y, inner='box', color = '#BEAED4')

    means = adata_sn.obs.groupby(x)[y].mean()


    plt.xlabel('')
    plt.ylabel(f'{y}')
    plt.title(f'{y} Detected per {x}')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.grid(False) 
    plt.savefig(f'figures/{x}_{y}_sn.pdf', dpi=300, bbox_inches="tight")
    plt.show()
    print(means)

In [ ]:
plotviolin('biobank_name','genes_detected')

In [ ]:
plotviolin('batch','genes_detected')

In [ ]:
plotviolin('batch','total_counts')

In [ ]:
plotviolin('biobank_name','total_counts')

In [ ]:
plotviolin('biobank_name','pct_counts_mt')

In [ ]:
plotviolin('batch','pct_counts_mt')

In [ ]:
adata_sn.obs['genes_detected'].mean()

# CC gene expression dotplot + QC plots

In [ ]:
adata_cc.obs['correct_ct'] = [cell_type_labels_corr.get(x, x) for x in adata_cc.obs['consensus_cell_type_corrected']]
adata_cc.obs

In [ ]:
adata_cc.obs['genes_detected'] = np.where(adata_cc.obs['gene_count'].notna(), adata_cc.obs['gene_count'], adata_cc.obs['n_genes'])
adata_cc.obs['genes_detected']


In [ ]:
def plotviolin(x,y):
    plt.figure(figsize=(5, 5))
    sns.violinplot(data=adata_cc.obs, x=x, y=y, inner='box', color="#7FC97F")

    means = adata_cc.obs.groupby(x)[y].mean()


    plt.xlabel('')
    plt.ylabel(f'{y}')
    plt.title(f'{y} Detected per {x}')
    plt.xticks(rotation=45, ha='right')
  #  plt.yscale('log')
    plt.tight_layout()
    plt.grid(False) 
    plt.savefig(f'figures/{x}_{y}_cc.pdf', dpi=300, bbox_inches="tight")
    plt.show()
    print(means)

In [ ]:
## CC QC plots for supplementary figure
plotviolin('biobank_name','genes_detected')
plotviolin('batch','genes_detected')
plotviolin('biobank_name','total_counts')
plotviolin('batch','total_counts')
plotviolin('batch','pct_counts_mt')
plotviolin('biobank_name','pct_counts_mt')

In [ ]:
adata_cc.X  = adata_cc.layers['log1p_norm']
from collections import defaultdict

marker_genes = {
    "Neurons": ["SYT1"],
    "Excitatory":["SLC17A7","SATB2"], 
    "L2/3 IT":["CUX2","CA10", "LAMA2"],
    "L4 IT":["RORB","TSHZ2","RMST","CPNE4"],
    "L5 IT": [ "LOC105374971","FOXP2","EPHA3"], 
    "L6 IT":["THEMIS","LOC105373893","CBLN2","PTPRK"],
    "L6 IT Car3":["SNTB1","NTNG2", "MCTP2"],
    "L6 CT":["SEMA3E","LOC105373592","MEIS2"], 
    "L6b":["CCN2","ZBTB7C","HS3ST4","TLE4"], 
   "L5/6 NP":["HTR2C","NPSR1-AS1"], 
    "L5 ET":["LOC101927745","COL24A1","COL5A2"], 
    "Inhibitory":["GAD1","GAD2"],
    "CGE":["ADARB2"],
    "Lamp5":["LAMP5","RELN","FGF13"], 
    "Sncg":["CXCL14","CNR1","RGS12"],
    "Vip":["VIP","FNDC1","THSD7A"], 
    "MGE":["LHX6"],
    "Sst":["SST","FBN2","GRIK1"],
    "Sst Chodl":["NPY", "CRHBP", "NOS1"], 
    "Pvalb":["PVALB","RPH3AL","ADAMTS17","NXPH1"], 
    "Astro":["AQP4", "GFAP"],
    "Oligo": ["MAG", "MOBP", "SOX10"],
    "OPC": ["PDGFRA","C1QL1"],
    "Micro-PVM":["CD74","RUNX1"],
    "Immune":["PTPRC","ARHGAP15", "IQGAP2", ], 
    "Endo/VLMC":["RBPMS","CLDN5","EPAS1"], 
}


##### check for duplicate genes
gene_to_groups = defaultdict(list)
for group, genes in marker_genes.items():
    for gene in genes:
        gene_to_groups[gene].append(group)

duplicates = {gene: groups for gene, groups in gene_to_groups.items() if len(groups) > 1}

if not duplicates:
    print("No duplicates found.")
else:
    print("Duplicates:")
    for gene, groups in sorted(duplicates.items()):
        print(f"{gene}: {', '.join(groups)}")


## plot
dp = sc.pl.dotplot(
    adata_cc,
    marker_genes,
    groupby="correct_ct",
    standard_scale="var",
    mean_only_expressed=False,
    swap_axes=False,
    categories_order=[
         'L2/3 IT',
         'L4 IT',
         'L5 IT A',
         'L5 IT B',
         'L6 IT',
         'L6 IT Car3',
         'L6 CT',
         'L6b',
         'L5/6 NP',
         'L5 ET',
         'Lamp5 Exc',
         'Lamp5',
         'Sncg',
         'Vip',
         'Sst',
         'Sst Chodl',
         'Pvalb',
         'Neurons unk',
         'Astro',
         'Oligo',
         'OPC',
         'Micro-PVM',
         'Immune',
        'VLMC',
        'Endo',
    ],

    show=False
)


ax = dp["mainplot_ax"]

# Make tick labels italic and larger
for l in ax.get_xticklabels():
    l.set_style("italic")
    l.set_fontsize(15)  

for l in ax.get_yticklabels():
    l.set_fontsize(15)  


ax.set_xlabel(ax.get_xlabel(), fontsize=15)
ax.set_ylabel(ax.get_ylabel(), fontsize=15)

plt.savefig('figures/cc_dotplot_marker_genes.pdf', bbox_inches='tight')

plt.show()



## UMAPS CC for supplementary

In [ ]:
def plot_and_save_umap(adata, color, filename, legend_loc='center left'):
    with plt.rc_context({"figure.figsize": (6, 6), "figure.dpi": 300, "figure.frameon": False}):
        ax = sc.pl.umap(adata, color=color, show=False, frameon=False, legend_loc=legend_loc)
        fig = ax.get_figure()
        
        if legend_loc is not None:
            plt.legend(fontsize=10, loc=legend_loc, bbox_to_anchor=(1, 0.5), frameon=False)
        
        plt.show()
        fig.savefig(f"figures/{filename}.pdf", bbox_inches='tight')


plot_and_save_umap(adata_cc, "batch", "cc_manuscript_umap_batch")
plot_and_save_umap(adata_cc, "primary_diagnosis", "cc_manuscript_umap_primary_diagnosis")
plot_and_save_umap(adata_cc, "sex", "cc_manuscript_umap_sex")
plot_and_save_umap(adata_cc, "biobank_name", "cc_manuscript_umap_biobank")
plot_and_save_umap(adata_cc, "vireo_assignment", "cc_manuscript_umap_vireo_assignment", legend_loc=None)
plot_and_save_umap(adata_cc, "region", "cc_manuscript_umap_region")
